# Evaluate a trained VLG-CBM -- portable Colab version

Nothing is hand-carried: code from GitHub, images from Kaggle, annotations and trained
weights from a GitHub Release, backbones from public hubs. Open in Colab, Run all.

**One-time setup:** you need Kaggle API credentials for the images. Get `kaggle.json` from
kaggle.com -> Settings -> API -> Create New Token, then either add `KAGGLE_USERNAME` /
`KAGGLE_KEY` to Colab's Secrets (key icon in the left sidebar) or upload `kaggle.json`
when the data cell prompts.

Runtime -> Change runtime type -> **T4 GPU**.

In [ ]:
!pip install -q open_clip_torch ftfy regex loguru "setuptools<81" kaggle

In [ ]:
import os, subprocess
REPO_DIR = "/content/VLG-CBM"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-q", "-b", "bioclip-birds525",
                    "https://github.com/aygovind/VLG-CBM.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("repo:", os.getcwd())

## Annotations + trained weights

From the GitHub Release -- ~60MB, no auth.

In [ ]:
import subprocess, os

REL = "https://github.com/aygovind/VLG-CBM/releases/download/birds525-eval-v1"

for name, dest in [("annotations-val.tar.gz", "."), ("models-birds525.tar.gz", ".")]:
    if not os.path.exists(name):
        subprocess.run(["wget", "-q", "--show-progress", f"{REL}/{name}"], check=True)
    subprocess.run(["tar", "xzf", name, "-C", dest], check=True)

print("annotations:", len(os.listdir("annotations/birds525_val")))
print("models:", sorted(os.listdir("saved_models")))

## Images

Pulled from Kaggle, then checked against a manifest of the exact files the annotations
were generated against. Annotations are keyed by ImageFolder *position*, so a different
revision of the dataset would silently misalign every box -- the assert catches that
instead of letting it through.

In [ ]:
import os, json, shutil, subprocess, hashlib

# Credentials: Colab Secrets first, else prompt for kaggle.json
if not (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
    try:
        from google.colab import userdata
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    except Exception:
        from google.colab import files
        print("Upload kaggle.json:")
        files.upload()
        creds = json.load(open("kaggle.json"))
        os.environ["KAGGLE_USERNAME"] = creds["username"]
        os.environ["KAGGLE_KEY"] = creds["key"]

if not os.path.isdir("datasets/birds525/val"):
    subprocess.run(["kaggle", "datasets", "download", "-d", "gpiosenka/100-bird-species",
                    "-p", "/content/kaggle_birds", "--unzip"], check=True)
    os.makedirs("datasets/birds525", exist_ok=True)
    src = "/content/kaggle_birds/valid"
    assert os.path.isdir(src), f"expected {src} in the Kaggle download; got {os.listdir('/content/kaggle_birds')[:10]}"
    if os.path.exists("datasets/birds525/val"):
        shutil.rmtree("datasets/birds525/val")
    shutil.move(src, "datasets/birds525/val")

from torchvision import datasets
man = json.load(open("concept_files/birds525_val_manifest.json"))
ds = datasets.ImageFolder("datasets/birds525/val")
rel = [os.path.relpath(p, "datasets/birds525/val") for p, _ in ds.samples]
got = hashlib.sha256("\n".join(rel).encode()).hexdigest()

if got != man["sha256"]:
    raise AssertionError(
        f"val split does not match the annotations.\n"
        f"  expected {man['n_images']} images / {man['n_classes']} classes (sha {man['sha256'][:12]})\n"
        f"  got      {len(rel)} images / {len(ds.classes)} classes (sha {got[:12]})\n"
        "The Kaggle dataset has likely been revised since the annotations were generated; "
        "the box overlays would be misaligned.")
print(f"images verified: {len(rel)} across {len(ds.classes)} classes")

In [ ]:
os.environ["DATASET_FOLDER"] = "/content/VLG-CBM/datasets"

import torch
import vlgcbm_analysis as va

print("device:", "cuda" if torch.cuda.is_available() else "cpu")
print("models:", ", ".join(va.list_models("birds525")))

In [ ]:
MODEL   = "bioclip"      # <-- swap me
DATASET = "birds525"
SPLIT   = "val"

run = va.load_run(va.run_dir(MODEL, DATASET))
res = va.evaluate(run, split=SPLIT)
print(run)
print("accuracy: {:.2f}%".format(res.accuracy * 100))

## 1. Sankey: concept -> class

In [ ]:
best, worst = va.best_worst_classes(run, res, k=5)
print("best :", best)
print("worst:", worst)

In [ ]:
va.sankey_static(run, [best[0], worst[0]],
                 weight_cutoff=0.05, max_per_class=12,
                 save_path="figures/sankey.png")

## 2. Single example

In [ ]:
CLASS = worst[0]
idx_all   = va.class_indices(res, CLASS)
idx_wrong = va.class_indices(res, CLASS, only="wrong")
print(f"{CLASS}: {len(idx_all)} images, {len(idx_wrong)} wrong")

In [ ]:
IDX = int(idx_wrong[0]) if len(idx_wrong) else int(idx_all[0])
va.explain_example(run, idx=IDX, split=SPLIT)

## 3. Same example, with boxes

In [ ]:
va.explain_with_boxes(run, idx=IDX, split=SPLIT, top_k=8)

## 4. Model comparison grid

In [ ]:
outs = va.evaluate_many(va.find_runs(), split=SPLIT, keep_concept_acts=True)
for o in outs:
    print(o)

In [ ]:
va.story_figure(outs, split=SPLIT, top_concepts=2,
                flag_mode="sufficiency",
                save_path="figures/qualitative_comparison.png")